# 🏭 มาตรฐานอุตสาหกรรม (มอก.) บังคับ
**Dataset:** 148 รายการ | 19 หมวดหมู่ | 3 หน่วยงานรับผิดชอบ
**Source:** สำนักงานมาตรฐานผลิตภัณฑ์อุตสาหกรรม (สมอ.)

**เนื้อหาใน Notebook นี้:** โค้ดถูกออกแบบมาให้อ่านและทำความเข้าใจง่าย
- **ส่วนที่ 1:** การจัดการข้อมูล (Data Preparation)
- **ส่วนที่ 2:** การสร้างกราฟสรุปข้อมูล 5 แบบ (Data Visualization)
- **ส่วนที่ 3:** การวิเคราะห์ข้อความด้วย AI 3 โมเดล (NLP & Machine Learning)

In [1]:
# ติดตั้งไลบรารีที่จำเป็น
!pip install pandas plotly scikit-learn openpyxl -q

In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.io import to_html
import warnings
import os

# ปิดการแจ้งเตือนที่ไม่จำเป็น
warnings.filterwarnings('ignore')

# 1. โหลดข้อมูลจากไฟล์ Excel
df = pd.read_excel('std-tisi-b stru.xlsx')
df.columns = df.columns.str.strip()

# เปลี่ยนชื่อคอลัมน์ภาษาไทยให้เป็นภาษาอังกฤษ
df.rename(columns={
    'เลขที่ มอก.': 'TIS_NO',
    'ชื่อ มอก.': 'TIS_NAME_TH',
    'ชื่อ มอก. ภาษาอังกฤษ': 'TIS_NAME_EN',
    'ขอบข่ายภาษาอังกฤษ': 'SCOPE_EN',
    'หมวดหมู่ มอก.': 'ICS_NO',
    'วันที่ประกาศในราชกิจจา': 'EFFECTIVE_DATE',
    'หน่วยงาน': 'AGENCY'
}, inplace=True)

# 2. เลือกคอลัมน์สำคัญมาใช้งาน
columns_to_keep = ['TIS_NO', 'TIS_NAME_TH', 'TIS_NAME_EN', 'SCOPE_EN', 'ICS_NO', 'EFFECTIVE_DATE', 'AGENCY']
df = df[columns_to_keep].copy()

# 3. สร้างคอลัมน์ใหม่ (Feature Engineering)
# สกัดหมวดหมู่หลักออกมาจากรหัส ICS (เอาแค่ตัวเลขชุดแรกก่อนเครื่องหมายจุด)
df['ics_main'] = df['ICS_NO'].astype(str).apply(lambda code: code.split('.')[0] if '.' in code else code)

# ดึงปี (Year) ออกมาจากวันที่บังคับใช้
df['EFFECTIVE_DATE'] = pd.to_datetime(df['EFFECTIVE_DATE'], errors='coerce')
df['year'] = df['EFFECTIVE_DATE'].dt.year.fillna(0).astype(int)

# 4. จัดกลุ่มหมวดหมู่หลักให้เป็นชื่อภาษาไทยที่เข้าใจง่าย
category_mapping = {
    '13': 'สิ่งแวดล้อมและความปลอดภัย', 
    '23': 'ของไหลและท่อ', 
    '29': 'วิศวกรรมไฟฟ้า',
    '43': 'ยานยนต์', 
    '77': 'โลหะวิทยา (เหล็ก)', 
    '91': 'วัสดุก่อสร้าง',
    '83': 'ยางและพลาสติก', 
    '97': 'เครื่องใช้ในบ้าน'
}
# หมวดไหนไม่มีชื่อใน mapping ก็ให้ใช้เป็นคำว่า 'หมวด XX' ไปก่อน
df['category_name'] = df['ics_main'].map(category_mapping).fillna('หมวด ' + df['ics_main'])

# 5. เติมคำอธิบาย (SCOPE_EN) ที่ว่างเปล่าด้วยชื่อเรื่องแทน เพื่อให้ AI มีข้อมูลใช้วิเคราะห์
df['text_for_ai'] = df['SCOPE_EN'].fillna(df['TIS_NAME_EN']).fillna('')

print(f'✅ โหลดข้อมูลเสร็จสิ้น จำนวนทั้งหมด: {len(df)} รายการ')
print(f'📌 จำนวนหมวดหมู่ทั้งหมด: {df["category_name"].nunique()} หมวด')
print(f'🏢 จำนวนหน่วยงาน: {df["AGENCY"].nunique()} หน่วยงาน')

✅ โหลดข้อมูลเสร็จสิ้น จำนวนทั้งหมด: 149 รายการ
📌 จำนวนหมวดหมู่ทั้งหมด: 20 หมวด
🏢 จำนวนหน่วยงาน: 4 หน่วยงาน


In [3]:
# กำหนดชุดสีสำหรับกราฟ
COLOR_PALETTE = [
    '#264653', '#2a9d8f', '#e9c46a', '#f4a261', '#e76f51', 
    '#118ab2', '#073b4c', '#06d6a0', '#ef476f', '#ffd166',
    '#8338ec', '#ff006e', '#3a86ff'
]

def get_color(index):
    # ฟังก์ชันสุ่มสีแบบวนลูป เพื่อไม่ให้ Error หากสีหมด
    return COLOR_PALETTE[index % len(COLOR_PALETTE)]

---
## 📊 กราฟที่ 1 — Treemap: หมวดหมู่ → หน่วยงาน
ดูสัดส่วนว่าหมวดหมู่ไหนมี มอก. เยอะสุด และแต่ละหมวดดูแลโดยหน่วยงานใด

In [4]:
# 1. นับจำนวน มอก. ตาม หมวดหมู่ และ หน่วยงาน
category_agency_counts = df.groupby(['category_name', 'AGENCY']).size().reset_index(name='count')
category_totals = df.groupby('category_name').size().reset_index(name='count')

# 2. สร้างโครงสร้างข้อมูลสำหรับ Treemap
node_ids = ['TISI']
labels = ['มาตรฐานบังคับทั้งหมด']
parents = ['']
values = [len(df)]
colors = ['rgba(0,0,0,0)']

# ชั้นที่ 1 (หมวดหมู่สินค้า)
for index, row in category_totals.iterrows():
    cat_name = row['category_name']
    node_ids.append(cat_name)
    labels.append(cat_name)
    parents.append('TISI')
    values.append(row['count'])
    colors.append(get_color(index))

# ชั้นที่ 2 (หน่วยงานที่ดูแล)
for index, row in category_agency_counts.iterrows():
    cat_name = row['category_name']
    agency = str(row['AGENCY'])
    
    node_ids.append(f"{cat_name}|{agency}")
    labels.append(agency)
    parents.append(cat_name)
    values.append(row['count'])
    
    # หาว่าหมวดหมู่นี้ใช้สีอะไร เพื่อให้สีหน่วยงานตรงกับสีหมวดหมู่หลัก
    parent_color = colors[node_ids.index(cat_name)]
    colors.append(parent_color)

# 3. วาดกราฟ
figure_treemap = go.Figure(go.Treemap(
    ids=node_ids, 
    labels=labels, 
    parents=parents, 
    values=values,
    marker=dict(colors=colors),
    branchvalues='total', 
    hovertemplate='<b>%{label}</b><br>จำนวน: %{value} ฉบับ<extra></extra>',
    textinfo='label+value'
))

figure_treemap.update_layout(
    title='Treemap: สัดส่วนหมวดหมู่ผลิตภัณฑ์ (แยกตามหน่วยงานที่รับผิดชอบ)',
    margin=dict(t=40, l=10, r=10, b=10), 
    height=500, 
    font=dict(family='Sarabun, sans-serif')
)
figure_treemap.show()

## 📊 กราฟที่ 2 — Timeline: ประวัติการประกาศบังคับใช้ มอก.

In [5]:
# กรองเอาเฉพาะข้อมูลที่มีระบุปีที่ชัดเจน (ปี > 1900)
valid_years_data = df[df['year'] > 1900]
standards_per_year = valid_years_data.groupby('year').size().reset_index(name='count')

figure_timeline = px.bar(
    standards_per_year, 
    x='year', 
    y='count',
    title='จำนวน มอก. บังคับ ที่ประกาศใช้แยกตามปี',
    labels={'year': 'ปี ค.ศ.', 'count': 'จำนวนมาตรฐาน (ฉบับ)'},
    template='plotly_white',
    color_discrete_sequence=['#2a9d8f']
)
figure_timeline.update_layout(height=400, font=dict(family='Sarabun, sans-serif'))
figure_timeline.show()

## 📊 กราฟที่ 3 — จัดอันดับ: จำนวน มอก. ตามหมวดหมู่

In [6]:
# จัดอันดับหมวดหมู่จากมากไปน้อย
top_categories = df['category_name'].value_counts().sort_values(ascending=True)

figure_bar = go.Figure(go.Bar(
    x=top_categories.values, 
    y=top_categories.index, 
    orientation='h',
    marker=dict(
        color=top_categories.values, 
        colorscale='Viridis'
    ),
    text=top_categories.values, 
    textposition='outside'
))

figure_bar.update_layout(
    title='จำนวน มอก. ตามหมวดหมู่ผลิตภัณฑ์',
    xaxis_title='จำนวน (ฉบับ)',
    template='plotly_white', 
    height=600, 
    font=dict(family='Sarabun, sans-serif'),
    margin=dict(l=150)
)
figure_bar.show()

## 📊 กราฟที่ 4 & 5 — เจาะลึกความสัมพันธ์ หน่วยงาน ↔ หมวดหมู่

In [7]:
# 4. กราฟ Sunburst (ดูภาพรวมหน่วยงานจากจุดศูนย์กลาง แตกออกเป็นหมวดหมู่)
figure_sunburst = px.sunburst(
    df.fillna({'AGENCY': 'ไม่ระบุ'}), 
    path=['AGENCY', 'category_name'], 
    title='Sunburst: หน่วยงานที่ดูแล → หมวดหมู่ผลิตภัณฑ์',
    color_discrete_sequence=px.colors.qualitative.Pastel
)
figure_sunburst.update_layout(height=500, font=dict(family='Sarabun, sans-serif'))
figure_sunburst.show()

# 5. กราฟ Heatmap (ตารางความถี่ ช่วยให้เห็นชัดเจนว่า กต. ไหน ทำอะไรเป็นหลัก)
heatmap_data = pd.crosstab(df['AGENCY'], df['category_name'])

figure_heatmap = px.imshow(
    heatmap_data.values, 
    x=heatmap_data.columns, 
    y=heatmap_data.index,
    text_auto=True, 
    color_continuous_scale='Blues',
    title='Heatmap: ตารางความหนาแน่น หน่วยงาน × หมวดหมู่'
)
figure_heatmap.update_layout(
    height=500, 
    font=dict(family='Sarabun, sans-serif'),
    xaxis_title='หมวดหมู่ผลิตภัณฑ์', 
    yaxis_title='หน่วยงานรับผิดชอบ'
)
figure_heatmap.show()

---
## 🤖 โมเดลที่ 1 — Text Classification: ทำนายหมวดหมู่จากข้อความ
**เป้าหมาย:** ให้ AI อ่านประโยคภาษาอังกฤษในคำอธิบายขอบข่ายมาตรฐาน แล้วสอนให้ทายว่า มอก. นั้นควรอยู่ในหมวดหมู่ใด
เราจะเปรียบเทียบ 2 อัลกอริทึม (Random Forest และ Logistic Regression) ว่าใครแม่นกว่ากัน

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix

# 1. คัดกรองข้อมูล: โมเดล Classification ต้องการตัวอย่างอย่างน้อย 2 ตัวต่อ 1 หมวดหมู่
category_counts = df['category_name'].value_counts()
valid_categories = category_counts[category_counts >= 2].index
model_data = df[df['category_name'].isin(valid_categories)].copy()

print(f"จำนวนหมวดหมู่ที่นำมาใช้ฝึก AI (ต้องมีอย่างน้อย 2 ฉบับ): {len(valid_categories)} หมวดหมู่")

# 2. แปลงข้อความ (Text) ให้กลายเป็นตัวเลข (TF-IDF Vector)
# TF-IDF คือการให้คะแนนความสำคัญของคำศัพท์ (คำที่พบบ่อยในเอกสารนี้ แต่ไม่พบบ่อยในเอกสารอื่น จะได้คะแนนสูง)
text_vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)

# ให้ AI อ่านข้อความแล้วสกัดคำศัพท์ 1,000 คำหลักออกมา
X_text_features = text_vectorizer.fit_transform(model_data['text_for_ai']).toarray()
y_targets = np.array(model_data['category_name'])

# 3. แข่งขันหาคนเก่งด้วยวิธี Cross-Validation (แบ่งสอบ 5 รอบ)
model_rf = RandomForestClassifier(n_estimators=200, random_state=42)
model_lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')

# หมายเหตุ: ข้อมูลเรามีน้อย บางรอบอาจจะมีคลาสโผล่ไปไม่ครบ จึงใช้ Cross-validation ธรรมดาไม่ได้ใช้ Stratified แบบเข้มงวด
scores_rf = cross_val_score(model_rf, X_text_features, y_targets, cv=5, scoring='accuracy')
scores_lr = cross_val_score(model_lr, X_text_features, y_targets, cv=5, scoring='accuracy')

# 4. วาดกราฟเปรียบเทียบคะแนน (Boxplot)
figure_cv = go.Figure()
figure_cv.add_trace(go.Box(y=scores_rf, name=f'Random Forest (เฉลี่ย {scores_rf.mean():.1%})', marker_color='#264653'))
figure_cv.add_trace(go.Box(y=scores_lr, name=f'Logistic Reg. (เฉลี่ย {scores_lr.mean():.1%})', marker_color='#e76f51'))
figure_cv.update_layout(
    title='การทดสอบความแม่นยำ (5-Fold Cross Validation)', 
    yaxis_title='Accuracy (ความแม่นยำ)',
    template='plotly_white', 
    height=400, 
    font=dict(family='Sarabun, sans-serif')
)
figure_cv.show()

# 5. ใช้โมเดลที่ชนะ (Logistic Regression) มาเรียนรู้เต็มรูปแบบ
X_train, X_test, y_train, y_test = train_test_split(X_text_features, y_targets, test_size=0.3, random_state=42)

model_lr.fit(X_train, y_train)
y_predicted = model_lr.predict(X_test)
accuracy = np.mean(y_predicted == y_test)
print(f"\n🏆 ความแม่นยำของ Logistic Regression บนชุดทดสอบ: {accuracy:.1%}")

# 6. สร้างแผนภาพ Confusion Matrix เพื่อดูจุดบอด (ว่าทายผิดเป็นหมวดไหน)
unique_classes = sorted(list(set(y_test) | set(y_predicted)))
confusion_mat = confusion_matrix(y_test, y_predicted, labels=unique_classes)

figure_cm_text = px.imshow(
    confusion_mat, 
    text_auto=True, 
    x=unique_classes, 
    y=unique_classes,
    labels={'x': 'คำทำนาย (Predicted)', 'y': 'ของจริง (Actual)'}, 
    color_continuous_scale='Greens',
    title=f'Confusion Matrix — หมวดหมู่ผลิตภัณฑ์ (Accuracy: {accuracy:.1%})'
)
figure_cm_text.update_layout(height=600, width=800, font=dict(family='Sarabun, sans-serif'))
figure_cm_text.show()

# 7. สกัดคำศัพท์ชี้เป็นชี้ตาย (Top Features) ที่ทำให้โมเดลตัดสินใจ
vocab = np.array(text_vectorizer.get_feature_names_out())
coefficients = model_lr.coef_[0]  # ดึงน้ำหนักคำศัพท์ของคลาสแรกสุด
top_indices = np.argsort(np.abs(coefficients))[-15:]  # เลือก 15 คำที่น้ำหนักแรงสุด

figure_features = go.Figure(go.Bar(
    x=np.abs(coefficients)[top_indices], 
    y=vocab[top_indices], 
    orientation='h',
    marker=dict(color='#2a9d8f')
))
figure_features.update_layout(
    title=f'15 คำศัพท์ที่บ่งชี้ว่า มอก. เป็นหมวด [{unique_classes[0]}] มากที่สุด',
    xaxis_title='ความสำคัญของคำศัพท์ (Importance Weight)',
    template='plotly_white', 
    height=500, 
    font=dict(family='Sarabun, sans-serif')
)
figure_features.show()

จำนวนหมวดหมู่ที่นำมาใช้ฝึก AI (ต้องมีอย่างน้อย 2 ฉบับ): 16 หมวดหมู่



🏆 ความแม่นยำของ Logistic Regression บนชุดทดสอบ: 68.2%


---
## 📊 โมเดลที่ 2 — Text Clustering (K-Means): จัดกลุ่ม มอก. อัตโนมัติ
**เป้าหมาย:** ปล่อยให้ AI จัดกลุ่ม มอก. 148 ฉบับ ขึ้นมาใหม่ **ตามเนื้อหาและคำศัพท์ที่คล้ายคลึงกัน** โดยไม่ต้องสนว่าสมอ. เคยจัดหมวดหมู่ไว้อย่างไร

In [9]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# 1. ใช้ข้อความทั้งหมดแปลงเป็นตัวเลข 1,000 คอลัมน์ (TF-IDF)
X_all_text = text_vectorizer.transform(df['text_for_ai']).toarray()

# 2. ให้ AI จัด มอก. เป็น 6 กลุ่ม (Clusters)
NUMBER_OF_TEXT_CLUSTERS = 6
kmeans_model = KMeans(n_clusters=NUMBER_OF_TEXT_CLUSTERS, random_state=42)
df['text_cluster'] = kmeans_model.fit_predict(X_all_text)

sil_score = silhouette_score(X_all_text, df['text_cluster'])

# 3. ย่อข้อมูล 1,000 คอลัมน์ (คำศัพท์ 1,000 คำ) ลงมาให้เหลือแค่แกน 2 มิติ (PCA)
pca_model = PCA(n_components=2, random_state=42)
pca_coords = pca_model.fit_transform(X_all_text)
df['pca1'] = pca_coords[:, 0]
df['pca2'] = pca_coords[:, 1]

# 4. สร้างแผนภาพ PCA Scatter วาดจุดแต่ละมาตรฐาน มอก.
figure_text_pca = go.Figure()
for cluster_id in range(NUMBER_OF_TEXT_CLUSTERS):
    cluster_data = df[df['text_cluster'] == cluster_id]
    figure_text_pca.add_trace(go.Scatter(
        x=cluster_data['pca1'], 
        y=cluster_data['pca2'], 
        mode='markers',
        name=f'Cluster {cluster_id}',
        marker=dict(size=10, color=get_color(cluster_id), line=dict(width=1, color='white')),
        text=cluster_data['TIS_NAME_TH'],
        hovertemplate='<b>%{text}</b><br>กลุ่ม: '+str(cluster_id)+'<extra></extra>'
    ))

figure_text_pca.update_layout(
    title=f'PCA — AI จัดกลุ่ม มอก. {NUMBER_OF_TEXT_CLUSTERS} กลุ่มตามข้อความ (Silhouette Score: {sil_score:.3f})',
    template='plotly_white', 
    height=500, 
    font=dict(family='Sarabun, sans-serif')
)
figure_text_pca.show()

# 5. หา Keyword ของแต่ละกลุ่ม ว่าแต่ละ Cluster พูดถึงเรื่องอะไร
cluster_centers = kmeans_model.cluster_centers_
vocab = np.array(text_vectorizer.get_feature_names_out())

keywords = []
for i in range(NUMBER_OF_TEXT_CLUSTERS):
    # หาคำศัพท์ที่มีคะแนนเฉลี่ยสูงสุด 5 อันดับแรกในกลุ่มนี้
    top_word_indices = np.argsort(cluster_centers[i])[-5:]
    top_words = vocab[top_word_indices][::-1]  # เรียงจากมากไปน้อย
    keywords.append(f"กลุ่ม {i}: " + ", ".join(top_words))

figure_keywords = go.Figure(go.Table(
    header=dict(values=['กลุ่ม (Cluster)', 'คำศัพท์สำคัญประจำกลุ่ม (Top Keywords)'], fill_color='#264653', font=dict(color='white', size=14)),
    cells=dict(values=[list(range(NUMBER_OF_TEXT_CLUSTERS)), [kw.split(': ')[1] for kw in keywords]], fill_color='#f0f2f5', height=40)
))
figure_keywords.update_layout(
    title='คีย์เวิร์ดประจำกลุ่มที่ AI จัดมาให้',
    height=400, 
    font=dict(family='Sarabun, sans-serif')
)
figure_keywords.show()

---
## 📝 โมเดลที่ 3 — NMF Topic Modeling: สกัดหัวข้อหลักใน มอก.
**เป้าหมาย:** วิเคราะห์ว่าจาก มอก. ทั้ง 148 ฉบับ มี "หัวเรื่อง (Topic)" หลักๆ อะไรซ่อนอยู่บ้าง คล้ายกับการทำ Text Summarization แบบสกัดประเด็น

In [10]:
from sklearn.decomposition import NMF

# 1. ใช้ NMF เพื่อสกัดหัวเรื่องออกมา 6 หัวเรื่อง
NUMBER_OF_TOPICS = 6
nmf_model = NMF(n_components=NUMBER_OF_TOPICS, random_state=42)
document_topics = nmf_model.fit_transform(X_all_text)
topic_words_matrix = nmf_model.components_

# 2. ดึง 10 คำหลักของแต่ละหัวเรื่อง (Topic)
topic_labels = []
heatmap_data = []
for i, topic_weights in enumerate(topic_words_matrix):
    top_indices = np.argsort(topic_weights)[-10:]
    top_words = vocab[top_indices][::-1]
    topic_labels.append(f'Topic {i+1}<br>({top_words[0]}, {top_words[1]})')
    heatmap_data.append(topic_weights[top_indices][::-1])

# 3. แผนภาพ Heatmap แสดงน้ำหนักของคำศัพท์ในแต่ละหัวเรื่อง
figure_nmf_heatmap = px.imshow(
    heatmap_data, 
    y=topic_labels,
    color_continuous_scale='Purples', 
    aspect='auto',
    title='Topic-Term Heatmap: น้ำหนักของคำศัพท์สำคัญในแต่ละหัวเรื่อง (Topic)'
)
figure_nmf_heatmap.update_layout(height=450, font=dict(family='Sarabun, sans-serif'))
figure_nmf_heatmap.show()

# 4. มอก. แต่ละฉบับเป็นของหัวเรื่องไหนมากที่สุด?
df['Dominant_Topic'] = document_topics.argmax(axis=1)
topic_distribution = df['Dominant_Topic'].value_counts().sort_index()

figure_nmf_dist = go.Figure(go.Bar(
    x=[f'Topic {i+1}' for i in topic_distribution.index], 
    y=topic_distribution.values,
    marker_color='#8338ec', 
    text=topic_distribution.values, 
    textposition='auto'
))
figure_nmf_dist.update_layout(
    title='จำนวน มอก. ในแต่ละหัวเรื่อง (Dominant Topic Distribution)',
    xaxis_title='หัวเรื่องหลัก', 
    yaxis_title='จำนวนเอกสาร (ฉบับ)',
    template='plotly_white', 
    height=400, 
    font=dict(family='Sarabun, sans-serif')
)
figure_nmf_dist.show()

---
## Export → สร้างไฟล์ HTML (tisi_viz.html)

In [11]:
CSS = """\
*{box-sizing:border-box}
body{font-family:'Sarabun','Segoe UI',sans-serif;background:#f8f9fa;margin:0;padding:0;color:#212529}
.page{max-width:1140px;margin:0 auto;padding:32px 24px 64px}
.hero{background:linear-gradient(135deg,#03071e 0%,#370617 30%,#6a040f 60%,#d00000 100%);border-radius:16px;padding:40px 48px;margin-bottom:24px;color:white;position:relative;overflow:hidden}
.hero::before{content:'🏭';position:absolute;right:40px;top:12px;font-size:100px;opacity:.15;line-height:1}
.hero h1{margin:0 0 6px;font-size:2.1em;font-weight:700;letter-spacing:-.5px}
.hero p{margin:0;opacity:.72;font-size:1em}
.badge{display:inline-block;background:rgba(255,255,255,.14);border:1px solid rgba(255,255,255,.25);border-radius:20px;padding:3px 13px;font-size:.83em;margin:12px 6px 0 0}
.stats{display:grid;grid-template-columns:repeat(4,1fr);gap:16px;margin-bottom:24px}
.stat-card{background:white;border-radius:12px;padding:20px 22px;box-shadow:0 1px 5px rgba(0,0,0,.08);border-left:4px solid var(--ac)}
.stat-card .num{font-size:1.9em;font-weight:700;color:var(--ac);line-height:1.1}
.stat-card .lbl{font-size:.84em;color:#6c757d;margin-top:5px}
.card{background:white;border-radius:12px;box-shadow:0 1px 5px rgba(0,0,0,.08);margin-bottom:24px;overflow:hidden}
.card-header{padding:16px 22px 0;font-size:.78em;font-weight:700;text-transform:uppercase;letter-spacing:.07em;color:#6c757d}
.grid-2{display:grid;grid-template-columns:1fr 1fr;gap:24px}
.section-title{font-size:1.3em;font-weight:700;color:#370617;margin:32px 0 16px;padding-left:12px;border-left:4px solid #d00000}
@media(max-width:768px){.stats{grid-template-columns:1fr 1fr}.grid-2{grid-template-columns:1fr}.hero h1{font-size:1.4em}.hero::before{display:none}}
"""

all_figures = {
    'tree': figure_treemap, 'tl': figure_timeline, 'bar': figure_bar, 
    'sun': figure_sunburst, 'heat': figure_heatmap,
    'cv': figure_cv, 'cm2': figure_cm_text, 'feat': figure_features, 
    'pca2': figure_text_pca, 'kwtbl': figure_keywords,
    'nmfh': figure_nmf_heatmap, 'nmfd': figure_nmf_dist
}

html_snippets = {}
for key, fig in all_figures.items():
    html_snippets[key] = to_html(fig, include_plotlyjs=False, full_html=False, config={'responsive': True})

HTML_TEMPLATE = f"""<!DOCTYPE html>
<html lang='th'>
<head>
<meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'>
<title>มาตรฐานบังคับ TISI — Dashboard + Models</title>
<link href='https://fonts.googleapis.com/css2?family=Sarabun:wght@400;600;700&display=swap' rel='stylesheet'>
<script src='https://cdn.plot.ly/plotly-2.35.2.min.js'></script>
<style>:root{{--ac:#6a040f}}{CSS}</style>
</head>
<body>
<div class='page'>

<div class='hero'>
  <h1>🏭 มาตรฐาน มอก. บังคับ</h1>
  <p>วิเคราะห์ข้อมูลรายชื่อมาตรฐานอุตสาหกรรมบังคับ · 5 Viz + 3 NLP Models</p>
  <span class='badge'>📄 {len(df)} ฉบับ</span>
  <span class='badge'>🏷️ {df['category_name'].nunique()} หมวดหมู่</span>
  <span class='badge'>🏢 {df['AGENCY'].nunique()} กองงาน</span>
</div>

<div class='stats'>
  <div class='stat-card' style='--ac:#03071e'><div class='num'>{len(df)}</div><div class='lbl'>มอก. บังคับ</div></div>
  <div class='stat-card' style='--ac:#6a040f'><div class='num'>{df['category_name'].nunique()}</div><div class='lbl'>หมวดหมู่</div></div>
  <div class='stat-card' style='--ac:#d00000'><div class='num'>{df['year'].max()}</div><div class='lbl'>ประกาศล่าสุดปี (ค.ศ.)</div></div>
  <div class='stat-card' style='--ac:#e85d04'><div class='num'>{df['AGENCY'].nunique()}</div><div class='lbl'>หน่วยงานดูแล</div></div>
</div>

<div class='section-title'>📊 Visualizations (กราฟสรุปข้อมูล)</div>
<div class='grid-2'>
  <div class='card'><div class='card-header'>VIZ 1 — Treemap: หมวดหมู่ → หน่วยงาน</div>{html_snippets['tree']}</div>
  <div class='card'><div class='card-header'>VIZ 2 — Timeline: มอก. ที่ประกาศรายปี</div>{html_snippets['tl']}</div>
</div>
<div class='card'><div class='card-header'>VIZ 3 — จำนวน มอก. ตามหมวดหมู่</div>{html_snippets['bar']}</div>
<div class='grid-2'>
  <div class='card'><div class='card-header'>VIZ 4 — Sunburst: หน่วยงาน → หมวดหมู่</div>{html_snippets['sun']}</div>
  <div class='card'><div class='card-header'>VIZ 5 — Heatmap: หน่วยงาน × หมวดหมู่</div>{html_snippets['heat']}</div>
</div>

<div class='section-title'>🤖 Model 1 — Text Classification (ความแม่นยำสูงสุด: {scores_lr.mean():.1%})</div>
<div class='card'><div class='card-header'>5-Fold CV: RF vs Logistic Regression</div>{html_snippets['cv']}</div>
<div class='grid-2'>
  <div class='card'><div class='card-header'>Confusion Matrix — Logistic Regression</div>{html_snippets['cm2']}</div>
  <div class='card'><div class='card-header'>Top 15 TF-IDF Features สำหรับหมวดแรก</div>{html_snippets['feat']}</div>
</div>

<div class='section-title'>📊 Model 2 — K-Means Text Clustering (คะแนนกลุ่ม: {sil_score:.3f})</div>
<div class='grid-2'>
  <div class='card'><div class='card-header'>PCA Scatter — การจัด {NUMBER_OF_TEXT_CLUSTERS} กลุ่มด้วย AI</div>{html_snippets['pca2']}</div>
  <div class='card'><div class='card-header'>Top Keywords ประจำแต่ละกลุ่ม</div>{html_snippets['kwtbl']}</div>
</div>

<div class='section-title'>📝 Model 3 — NMF Topic Modeling ({NUMBER_OF_TOPICS} Topics)</div>
<div class='card'><div class='card-header'>Topic-Term Heatmap (น้ำหนักคำในแต่ละหัวเรื่อง)</div>{html_snippets['nmfh']}</div>
<div class='card'><div class='card-header'>จำนวนเอกสารในแต่ละหัวเรื่องหลัก</div>{html_snippets['nmfd']}</div>

</div></body></html>
"""

with open('tisi_viz.html', 'w', encoding='utf-8') as html_file:
    html_file.write(HTML_TEMPLATE)

print(f'✅ บันทึกไฟล์ tisi_viz.html สำเร็จแล้ว (ขนาด {os.path.getsize("tisi_viz.html")/1024:.0f} KB)')

✅ บันทึกไฟล์ tisi_viz.html สำเร็จแล้ว (ขนาด 144 KB)
